In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

batch_size, num_steps = 32, 35

data = d2l.TimeMachine(batch_size=batch_size, num_steps=num_steps)

vocab = data.vocab

train_iter = data.get_dataloader(train=True)

In [ ]:
def get_params(num_inputs, num_hiddens, num_outputs, device):
    params = []

    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01

    def three():
        return (normal((num_inputs, num_hiddens)), normal((num_hiddens, num_hiddens)), torch.zeros(num_hiddens, device=device))

    W_xh1, W_hh1, b_h1 = three()
    W_xh2, W_hh2, b_h2 = three()

    W_hq = torch.randn(size=(num_hiddens, num_outputs), device=device)
    b_q = torch.zeros(num_outputs, device=device)

    params = [W_xh1, W_hh1, b_h1, W_xh2, W_hh2, b_h2, W_hq, b_q]

    for param in params:
        param.requires_grad_(True)

    return params

In [ ]:
def rnn_2layer(inputs, state, params):
    (H1, H2) = state
    [W_xh1, W_hh1, b_h1, W_xh2, W_hh2, b_h2, W_hq, b_q] = params

    outputs = []

    for X in inputs:
        H1 = H1 @ W_hh1 + X @ W_xh1 + b_h1
        H2 = H2 @ W_hh2 + H1 @ W_xh2 + b_h2

        Y = H2 @ W_hq + b_q

        outputs.append(Y)

    return torch.cat(outputs, dim=0), (H1, H2)